# M3L4 E09 — LangGraph router + Langfuse
### Modulo 3 · Lecture 4 · Construccion, pruebas y trazabilidad de agentes en produccion

---

## Que necesitas saber antes

| Modulo | Concepto | Por que lo necesitas aca |
|---|---|---|
| M3L4 E08 | StateGraph, nodos, compilacion, CallbackHandler | Aca construimos un grafo con multiples nodos y routing condicional |
| M3L4 E05 | Router v2, multi-intent, clarification | El router v2 determina que nodo del grafo ejecutar |
| M3L4 E03 | Misclassification | Langfuse muestra en que nodo termino cada consulta |
| LangGraph | `add_conditional_edges()`, mapeo de rutas | Para routing condicional desde router_node a los agentes |

---

## Definiciones clave

| Concepto | Definicion simple | Como aparece en este notebook |
|---|---|---|
| **Router node** | Nodo que clasifica la query y decide que agente ejecutar | `router_node()` llama a `route_query_v2()` |
| **Conditional edges** | Aristas que dependen del estado para elegir destino | `add_conditional_edges('router_node', route_to_node, mapping)` |
| **Mapping** | Diccionario de posibles intents a nombres de nodos | `{'hr': 'hr_node', 'it': 'it_node', ...}` |
| **Nodo agente** | Nodo especialista que responde segun su dominio | `hr_node()`, `it_node()`, `finance_node()`, `legal_node()`, `general_node()` |
| **AgentState** | Estado compartido entre nodos: query, intent, response | `TypedDict` con 3 campos |

---

## Arquitectura

```
START
  |
  v
router_node (clasifica intent con route_query_v2)
  |
  |  conditional_edges (segun state['intent'])
  |
  +---> hr_node       -> END
  +---> it_node       -> END
  +---> finance_node  -> END
  +---> legal_node    -> END
  +---> general_node  -> END
```

**Objetivo del ejercicio:** crear un router multiagente en LangGraph y ver en Langfuse exactamente que nodo se ejecuto para cada consulta.

## Paso 1 — Instalacion

| Libreria | Que hace |
|---|---|
| `langfuse` | SDK de Langfuse para tracing |
| `langchain` | Base para integraciones |
| `langchain-openai` | ChatOpenAI (aunque no se usa directamente en este ejercicio) |
| `langgraph` | StateGraph con routing condicional |

```python
!pip install -q langfuse langchain langchain-openai langgraph
```

In [ ]:
!pip install -q langfuse langchain langchain-openai langgraph
print('Instalacion completa.')

In [ ]:
import os
from getpass import getpass

os.environ['LANGFUSE_PUBLIC_KEY'] = getpass('Langfuse Public Key: ')
os.environ['LANGFUSE_SECRET_KEY'] = getpass('Langfuse Secret Key: ')
os.environ['LANGFUSE_BASE_URL']   = 'https://cloud.langfuse.com'
os.environ['OPENAI_API_KEY']      = getpass('OpenAI API Key: ')
print('Credenciales OK.')

## Paso 2 — Imports

| Import | Que hace |
|---|---|
| `from typing_extensions import TypedDict` | Define el estado tipado del grafo |
| `from langgraph.graph import StateGraph, END` | Grafo de estado y nodo terminal |
| `from langfuse.langchain import CallbackHandler` | Interceptor de Langfuse para tracing |

A diferencia de E08, no necesitamos `ChatOpenAI` ni `HumanMessage` porque los nodos agente devuelven respuestas fijas (sin LLM).

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END
from langfuse.langchain import CallbackHandler

print('Imports OK.')

## Paso 3 — Router v2 (reutilizado de E05)

Misma funcion `route_query_v2()` que usamos en E05 y E06. Clasifica consultas en 6 intents: hr, it, finance, legal, multi_intent, clarification, general.

In [ ]:
def route_query_v2(query: str) -> str:
    q = query.lower()
    hr_kw      = ['vacaciones', 'licencia', 'recibo', 'nomina', 'rrhh']
    it_kw      = ['vpn', 'error', 'app', 'laptop', 'wifi', 'login', 'contrasena']
    finance_kw = ['factura', 'pago', 'reembolso', 'gasto', 'cobro', 'comprobante', 'salario']
    legal_kw   = ['contrato', 'legal', 'confidencialidad', 'nda', 'acuerdo']
    detected = []
    if any(w in q for w in hr_kw):      detected.append('hr')
    if any(w in q for w in it_kw):      detected.append('it')
    if any(w in q for w in finance_kw): detected.append('finance')
    if any(w in q for w in legal_kw):   detected.append('legal')
    if len(detected) > 1:   return 'multi_intent'
    if len(detected) == 1:  return detected[0]
    if len(q.split()) <= 2: return 'clarification'
    return 'general'

print('Router v2 listo.')

## Paso 4 — Estado del grafo

El estado `AgentState` es un diccionario con 3 campos que fluyen entre nodos:

```python
class AgentState(TypedDict):
    query: str        # la consulta original del usuario
    intent: str       # intent detectado por router_node
    response: str     # respuesta del agente especialista
```

**Flujo del estado:**

```
START:  {'query': '...', 'intent': '', 'response': ''}
    |
router_node:  {'intent': 'finance'}  (solo actualiza 'intent')
    |
finance_node: {'response': '...'}    (solo actualiza 'response')
    |
END:    {'query': '...', 'intent': 'finance', 'response': '...'}
```

In [ ]:
class AgentState(TypedDict):
    query: str
    intent: str
    response: str

print('AgentState definido.')

## Paso 5 — TODO: Nodos del sistema

Implementa los 6 nodos. Cada nodo recibe el estado y retorna un dict con SOLO los campos que actualiza.

### router_node(state)

| Parametro | Tipo | Que es |
|---|---|---|
| `state` | `AgentState` | Estado actual con `query` |
| **Retorna** | `dict` | `{'intent': route_query_v2(state['query'])}` |

### Nodos agente (hr_node, it_node, etc.)

Cada uno retorna `{'response': '...'}` con un mensaje fijo del agente.

In [ ]:
def router_node(state: AgentState) -> dict:
    """
    Detecta el intent de la query usando route_query_v2.
    Retorna solo {'intent': intent}
    """
    # TODO
    pass

def hr_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de HR
    pass

def it_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de IT
    pass

def finance_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de Finance
    pass

def legal_node(state: AgentState) -> dict:
    # TODO: retornar respuesta de Legal
    pass

def general_node(state: AgentState) -> dict:
    # TODO: retornar respuesta general
    pass

print('Nodos definidos.')

## Paso 6 — TODO: Funcion de routing condicional

`route_to_node(state)` es la funcion que LangGraph usa para decidir que nodo ejecutar despues de `router_node`.

```python
def route_to_node(state: AgentState) -> str:
    # Mapea state['intent'] al nombre del nodo destino
    # {'hr': 'hr_node', 'it': 'it_node', 'finance': 'finance_node',
    #  'legal': 'legal_node', 'multi_intent': 'general_node',
    #  'clarification': 'general_node', 'general': 'general_node'}
```

**Por que importa:** esta funcion es el "cerebro" del routing. Si falta un mapeo, LangGraph lanza error.

In [ ]:
def route_to_node(state: AgentState) -> str:
    """
    Mapea state['intent'] al nombre del nodo destino.
    Cualquier intent no reconocido -> 'general_node'
    """
    # TODO
    pass

print('Funcion condicional definida.')

## Paso 7 — TODO: Compilar el grafo

```python
builder = StateGraph(AgentState)
builder.add_node('router_node', router_node)
builder.add_node('hr_node', hr_node)
# ... agregar todos los nodos

builder.set_entry_point('router_node')

builder.add_conditional_edges(
    'router_node',
    route_to_node,
    {
        'hr_node': 'hr_node',
        'it_node': 'it_node',
        # ... mapear todos los retornos a sus nodos
    }
)

for node in ['hr_node', 'it_node', 'finance_node', 'legal_node', 'general_node']:
    builder.add_edge(node, END)

graph = builder.compile()
```

In [ ]:
# TODO: construir el StateGraph con AgentState
# 1. Agregar todos los nodos
# 2. set_entry_point('router_node')
# 3. add_conditional_edges de router_node usando route_to_node
# 4. add_edge de cada nodo especialista a END
# 5. Compilar

graph = None  # reemplazar
print('Grafo compilado.')

In [ ]:
if graph:
    print(graph.get_graph().draw_mermaid())

## Paso 8 — Ejecutar con Langfuse

Cada ejecucion del grafo produce:

- **1 Trace** por cada query
- **2 Spans**: `router_node` + `{intent}_node`
- **metadata** con tags para filtrar en Langfuse

In [ ]:
queries = [
    'Como solicito mis dias de vacaciones?',
    'Mi VPN no conecta desde ayer',
    'Necesito ver mi factura del mes pasado',
    'Necesito el contrato de confidencialidad actualizado',
    'ayuda'
]

for i, q in enumerate(queries):
    langfuse_handler = CallbackHandler()

    # TODO: invocar el grafo con:
    # - input: {'query': q, 'intent': '', 'response': ''}
    # - config con callbacks y metadata (tags: ['m3l4', 'router-demo'], user_id: f'student-{i}')
    output = None  # reemplazar

    if output:
        print(f'Query: {q[:45]}')
        print(f'  Intent: {output["intent"]} | Respuesta: {output["response"][:60]}...')
        print()

In [ ]:
if graph:
    lf = CallbackHandler()
    r = graph.invoke({'query': 'No puedo ver mi factura', 'intent': '', 'response': ''},
                     config={'callbacks': [lf]})
    assert r['intent'] == 'finance', f"Esperaba finance, obtuvo: {r['intent']}"
    assert len(r['response']) > 5
    print('Checks E09 OK')

## Errores comunes

| Error | Causa | Como detectarlo |
|---|---|---|
| `ConditionalEdge` no encuentra mapping | Faltan intents en el diccionario de `add_conditional_edges` | LangGraph lanza `ValueError` |
| `graph` es `None` | No compilar con `graph.compile()` | Los asserts fallan |
| `intent` queda vacio | `router_node()` no retorna `{'intent': ...}` | Las queries no se rutean |
| Todos los nodos retornan None | Los nodos agente no tienen `return` | `output['response']` es `None` |
| No aparece en Langfuse | Olvidar pasar `config={'callbacks': [handler]}` | Ejecuta bien pero sin trace |

## Sintesis

### Que construiste

| Componente | Descripcion |
|---|---|
| `router_node` | Clasifica la query usando route_query_v2 |
| 5 nodos agente | Respuestas fijas por dominio (hr, it, finance, legal, general) |
| `route_to_node()` | Funcion condicional que mapea intent a nodo |
| Grafo con routing | LangGraph con conditional_edges |
| Langfuse traces | Cada ejecucion queda registrada con tags y spans |

### Diferencia con E08

| Aspecto | E08 (1 nodo) | E09 (multiples nodos) |
|---|---|---|
| Estructura | `START -> chatbot -> END` | `START -> router -> [hr/it/finance/legal/general] -> END` |
| Routing | Lineal | Condicional (segun intent) |
| Spans en Langfuse | 1 span (chatbot) | 2 spans (router + agente) |
| Estado | Solo messages | query, intent, response |

### Relacion con otros ejercicios

| Ejercicio | Conexion con E09 |
|---|---|
| **E10** | Supervisor: patron con mas control y deteccion de loops |
| **E11** | Golden dataset sobre LangGraph: evaluar el grafo completo |
| **E12** | Ciclo de mejora: diagnosticar fallas y mejorar el router |